In [110]:
import nflreadpy as nfl
import pandas as pd
import pyarrow
import kagglehub
from kagglehub import KaggleDatasetAdapter
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest
from scipy.stats import ttest_ind, mannwhitneyu

In [27]:
team_stats_2024 = nfl.load_team_stats(
    seasons=2024,
    summary_level="week"
)

print(team_stats_2024)
print(team_stats_2024.columns)
print(team_stats_2024.shape)

shape: (570, 138)
┌────────┬──────┬──────┬─────────────┬───┬─────────────┬──────────────┬──────────────┬─────────────┐
│ season ┆ week ┆ team ┆ season_type ┆ … ┆ pt_returned ┆ pt_return_ya ┆ pt_return_td ┆ pt_net_yard │
│ ---    ┆ ---  ┆ ---  ┆ ---         ┆   ┆ ---         ┆ rds          ┆ s            ┆ s           │
│ i32    ┆ i32  ┆ str  ┆ str         ┆   ┆ i32         ┆ ---          ┆ ---          ┆ ---         │
│        ┆      ┆      ┆             ┆   ┆             ┆ i32          ┆ i32          ┆ i32         │
╞════════╪══════╪══════╪═════════════╪═══╪═════════════╪══════════════╪══════════════╪═════════════╡
│ 2024   ┆ 1    ┆ ARI  ┆ REG         ┆ … ┆ 1           ┆ 7            ┆ 0            ┆ 71          │
│ 2024   ┆ 1    ┆ ATL  ┆ REG         ┆ … ┆ 4           ┆ 47           ┆ 0            ┆ 163         │
│ 2024   ┆ 1    ┆ BAL  ┆ REG         ┆ … ┆ 0           ┆ 0            ┆ 0            ┆ 75          │
│ 2024   ┆ 1    ┆ BUF  ┆ REG         ┆ … ┆ 1           ┆ 6            ┆ 0

In [28]:
print(team_stats_2024["game_id"])
print(team_stats_2024["season_type"])

shape: (570,)
Series: 'game_id' [str]
[
	"2024_01_ARI_BUF"
	"2024_01_PIT_ATL"
	"2024_01_BAL_KC"
	"2024_01_ARI_BUF"
	"2024_01_CAR_NO"
	…
	"2024_21_BUF_KC"
	"2024_21_WAS_PHI"
	"2024_21_WAS_PHI"
	"2024_22_KC_PHI"
	"2024_22_KC_PHI"
]
shape: (570,)
Series: 'season_type' [str]
[
	"REG"
	"REG"
	"REG"
	"REG"
	"REG"
	…
	"POST"
	"POST"
	"POST"
	"POST"
	"POST"
]


In [29]:

team_stats_full = nfl.load_team_stats(
    seasons=list(range(2006, 2025)),
    summary_level="week"
)

In [30]:
keep_col = [
    # Identifiers
    'season',
    'week',
    'team',
    'season_type',
    'game_id',
    'opponent_team',
    # Passing
    'completions',
    'attempts',
    'passing_yards',
    'passing_tds',
    'passing_interceptions',
    'sacks_suffered',
    'passing_air_yards',
    'passing_yards_after_catch',
    'passing_first_downs',
    'passing_epa',
    'passing_cpoe',
    # Rushing
    'carries',
    'rushing_yards',
    'rushing_tds',
    'rushing_first_downs',
    'rushing_epa',
    # Defense
    'def_tackles_solo',
    'def_tackles_with_assist',
    'def_tackle_assists',
    'def_tackles_for_loss',
    'def_fumbles_forced',
    'def_sacks',
    'def_sack_yards',
    'def_qb_hits',
    'def_interceptions',
    'def_pass_defended',
    'def_tds',
    'def_fumbles',
    'def_safeties',
    # Turnovers / penalties
    'fumbles_total',
    'fumbles_lost_total',
    'penalties',
    'penalty_yards',
    # Special teams
    'fg_made',
    'fg_att',
    'fg_missed',
    'fg_blocked',
    'fg_long',
    'fg_pct',
    'punt_returns',
    'punt_return_yards',
    'kickoff_returns',
    'kickoff_return_yards'
]

In [31]:
team_stats_w_post = team_stats_full.select(keep_col).to_pandas()

In [33]:
team_stats = team_stats_w_post[
    team_stats_w_post["season_type"] == "REG"
].copy()

In [37]:
print("Shape:", team_stats.shape)

print("\nSeasons:")
print(team_stats["season"].unique())

print("\nGames by season:")
print(
    team_stats
    .groupby("season")["game_id"]
    .nunique()
)

print("\nRows / game:")
print(
    team_stats
    .groupby("game_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nMissing vals:")
print(
    team_stats
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Shape: (9854, 49)

Seasons:
[2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019
 2020 2021 2022 2023 2024]

Games by season:
season
2006    256
2007    256
2008    256
2009    256
2010    256
2011    256
2012    256
2013    256
2014    256
2015    256
2016    256
2017    256
2018    256
2019    256
2020    256
2021    272
2022    271
2023    272
2024    272
Name: game_id, dtype: int64

Rows / game:
2    4927
Name: count, dtype: int64

Missing vals:
fg_long                      1812
fg_pct                       1261
passing_cpoe                   11
season                          0
week                            0
opponent_team                   0
completions                     0
season_type                     0
team                            0
passing_yards                   0
passing_tds                     0
sacks_suffered                  0
passing_interceptions           0
passing_air_yards               0
passing_yards_after_catch       0
attempts          

In [38]:
team_stats[
    team_stats["passing_cpoe"].isna()
][
    [
        "season",
        "week",
        "team",
        "opponent_team",
        "game_id",
        "completions",
        "attempts",
        "passing_yards",
        "passing_cpoe"
    ]
]

,season,week,team,opponent_team,game_id,completions,attempts,passing_yards,passing_cpoe
106,2006,4,KC,SF,2006_04_SF_KC,18,23,208,NaN
117,2006,4,SF,KC,2006_04_SF_KC,13,25,92,NaN
187,2006,7,KC,LAC,2006_07_SD_KC,15,27,232,NaN
188,2006,7,LAC,KC,2006_07_SD_KC,26,44,267,NaN
213,2006,8,KC,SEA,2006_08_SEA_KC,17,25,312,NaN
224,2006,8,SEA,KC,2006_08_SEA_KC,15,30,198,NaN
306,2006,11,LV,KC,2006_11_OAK_KC,15,25,194,NaN
386,2006,14,BAL,KC,2006_14_BAL_KC,21,27,283,NaN
399,2006,14,KC,BAL,2006_14_BAL_KC,15,27,178,NaN
494,2006,17,JAX,KC,2006_17_JAX_KC,23,40,306,NaN


In [43]:
betting_data = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "tobycrabtree/nfl-scores-and-betting-data",
    "spreadspoke_scores.csv"
)

print("Shape:", betting_data.shape)
print("\nColumns:")
print(betting_data.columns.tolist())

print("\nFirst 5 rows:")
print(betting_data.head())

100%|██████████| 1.52M/1.52M [00:01<00:00, 1.46MB/s]

Shape: (14371, 17)

Columns:
['schedule_date', 'schedule_season', 'schedule_week', 'schedule_playoff', 'team_home', 'score_home', 'score_away', 'team_away', 'team_favorite_id', 'spread_favorite', 'over_under_line', 'stadium', 'stadium_neutral', 'weather_temperature', 'weather_wind_mph', 'weather_humidity', 'weather_detail']

First 5 rows:
  schedule_date  schedule_season schedule_week  schedule_playoff  \
0      9/2/1966             1966             1             False   
1      9/3/1966             1966             1             False   
2      9/4/1966             1966             1             False   
3      9/9/1966             1966             2             False   
4     9/10/1966             1966             1             False   

            team_home  score_home  score_away        team_away  \
0      Miami Dolphins          14          23  Oakland Raiders   
1      Houston Oilers          45           7   Denver Broncos   
2  San Diego Chargers          27           7    Buf

In [46]:
betting = betting_data[
    (betting_data["schedule_season"] >= 2006) &
    (betting_data["schedule_season"] <= 2024) &
    (betting_data["schedule_playoff"] == False)
].copy()

print("data shape:", betting.shape)

print("\nGames by season:")
print(
    betting
    .groupby("schedule_season")
    .size()
)

data shape: (4927, 17)

Games by season:
schedule_season
2006    256
2007    256
2008    256
2009    256
2010    256
2011    256
2012    256
2013    256
2014    256
2015    256
2016    256
2017    256
2018    256
2019    256
2020    256
2021    272
2022    271
2023    272
2024    272
dtype: int64


In [47]:
print("Teams in team_stats:")
print(sorted(team_stats["team"].unique()))

print("\nHome teams in betting data:")
print(sorted(betting["team_home"].unique()))

print("\nAway teams in betting data:")
print(sorted(betting["team_away"].unique()))

Teams in team_stats:
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

Home teams in betting data:
['Arizona Cardinals', 'Atlanta Falcons', 'Baltimore Ravens', 'Buffalo Bills', 'Carolina Panthers', 'Chicago Bears', 'Cincinnati Bengals', 'Cleveland Browns', 'Dallas Cowboys', 'Denver Broncos', 'Detroit Lions', 'Green Bay Packers', 'Houston Texans', 'Indianapolis Colts', 'Jacksonville Jaguars', 'Kansas City Chiefs', 'Las Vegas Raiders', 'Los Angeles Chargers', 'Los Angeles Rams', 'Miami Dolphins', 'Minnesota Vikings', 'New England Patriots', 'New Orleans Saints', 'New York Giants', 'New York Jets', 'Oakland Raiders', 'Philadelphia Eagles', 'Pittsburgh Steelers', 'San Diego Chargers', 'San Francisco 49ers', 'Seattle Seahawks', 'St. Louis Rams', 'Tampa Bay Buccaneers', 'Tennessee Titans', 'Washington Commanders', 'Washington 

In [48]:

team_name_map = {
    "Arizona Cardinals": "ARI",
    "Atlanta Falcons": "ATL",
    "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF",
    "Carolina Panthers": "CAR",
    "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN",
    "Cleveland Browns": "CLE",
    "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN",
    "Detroit Lions": "DET",
    "Green Bay Packers": "GB",
    "Houston Texans": "HOU",
    "Indianapolis Colts": "IND",
    "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC",
    "Las Vegas Raiders": "LV",
    "Los Angeles Chargers": "LAC",
    "Los Angeles Rams": "LA",
    "Miami Dolphins": "MIA",
    "Minnesota Vikings": "MIN",
    "New England Patriots": "NE",
    "New Orleans Saints": "NO",
    "New York Giants": "NYG",
    "New York Jets": "NYJ",
    "Philadelphia Eagles": "PHI",
    "Pittsburgh Steelers": "PIT",
    "San Francisco 49ers": "SF",
    "Seattle Seahawks": "SEA",
    "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    "Oakland Raiders": "LV",
    "San Diego Chargers": "LAC",
    "St. Louis Rams": "LA",
    "Washington Redskins": "WAS",
    "Washington Football Team": "WAS",
    "Washington Commanders": "WAS"
}
betting["team_home"] = betting["team_home"].map(team_name_map)
betting["team_away"] = betting["team_away"].map(team_name_map)

In [50]:
print("Unmapped home teams:")
print(betting["team_home"].isna().sum())

print("Unmapped away teams:")
print(betting["team_away"].isna().sum())

print(sorted(betting["team_home"].dropna().unique()))

Unmapped home teams:
0
Unmapped away teams:
0
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


In [51]:
betting["game_id"] = (
    betting["schedule_season"].astype(str)
    + "_"
    + betting["schedule_week"].astype(str).str.zfill(2)
    + "_"
    + betting["team_away"]
    + "_"
    + betting["team_home"]
)

In [55]:
print("betting games:", betting["game_id"].nunique())
print("team_stats games:", team_stats["game_id"].nunique())
print(
    "Matching games:",
    betting["game_id"].isin(team_stats["game_id"]).sum()
)
print(
    "Team stats games with betting data:",
    team_stats["game_id"].isin(betting["game_id"]).sum()
)

betting games: 4927
team_stats games: 4927
Matching games: 4395
Team stats games with betting data: 8790


In [56]:
unmatched_betting = betting[
    ~betting["game_id"].isin(team_stats["game_id"])
].copy()

print("Unmatched betting games:", len(unmatched_betting))

print(
    unmatched_betting[
        ["schedule_season", "schedule_week",
         "team_away", "team_home", "game_id"]
    ].head(20)
)

Unmatched betting games: 532
      schedule_season schedule_week team_away team_home          game_id
8951             2006             1       DEN        LA   2006_01_DEN_LA
8954             2006             1       LAC        LV   2006_01_LAC_LV
8957             2006             2        LV       BAL   2006_02_LV_BAL
8968             2006             2       TEN       LAC  2006_02_TEN_LAC
8969             2006             2        LA        SF    2006_02_LA_SF
8972             2006             3        LA       ARI   2006_03_LA_ARI
8987             2006             4       LAC       BAL  2006_04_LAC_BAL
8995             2006             4       CLE        LV   2006_04_CLE_LV
8996             2006             4       DET        LA   2006_04_DET_LA
9003             2006             5        LA        GB    2006_05_LA_GB
9011             2006             5       PIT       LAC  2006_05_PIT_LAC
9012             2006             5        LV        SF    2006_05_LV_SF
9017             2006 

In [57]:
unmatched_stats = team_stats[
    ~team_stats["game_id"].isin(betting["game_id"])
].copy()

print("Unmatched team stats rows:", len(unmatched_stats))

print(
    unmatched_stats[
        ["season", "week", "team", "opponent_team", "game_id"]
    ].head(20)
)

Unmatched team stats rows: 1064
     season  week team opponent_team          game_id
9      2006     1  DEN            LA  2006_01_DEN_STL
16     2006     1   LA           DEN  2006_01_DEN_STL
17     2006     1  LAC            LV   2006_01_SD_OAK
18     2006     1   LV           LAC   2006_01_SD_OAK
34     2006     2  BAL            LV  2006_02_OAK_BAL
48     2006     2   LA            SF   2006_02_STL_SF
49     2006     2  LAC           TEN   2006_02_TEN_SD
50     2006     2   LV           BAL  2006_02_OAK_BAL
60     2006     2   SF            LA   2006_02_STL_SF
62     2006     2  TEN           LAC   2006_02_TEN_SD
64     2006     3  ARI            LA  2006_03_STL_ARI
78     2006     3   LA           ARI  2006_03_STL_ARI
94     2006     4  BAL           LAC   2006_04_SD_BAL
99     2006     4  CLE            LV  2006_04_CLE_OAK
101    2006     4  DET            LA  2006_04_DET_STL
107    2006     4   LA           DET  2006_04_DET_STL
108    2006     4  LAC           BAL   2006_04_SD_

In [58]:
game_id_map = {
    "STL": "LA",
    "SD": "LAC",
    "OAK": "LV"
}

team_stats["game_id_normalized"] = (
    team_stats["game_id"]
    .str.split("_")
    .apply(
        lambda x: "_".join([
            x[0],
            x[1],
            game_id_map.get(x[2], x[2]),
            game_id_map.get(x[3], x[3])
        ])
    )
)

In [59]:
print(
    team_stats[
        ["game_id", "game_id_normalized"]
    ].drop_duplicates().head(20)
)

            game_id game_id_normalized
0    2006_01_SF_ARI     2006_01_SF_ARI
1   2006_01_ATL_CAR    2006_01_ATL_CAR
2    2006_01_BAL_TB     2006_01_BAL_TB
3    2006_01_BUF_NE     2006_01_BUF_NE
5    2006_01_CHI_GB     2006_01_CHI_GB
6    2006_01_CIN_KC     2006_01_CIN_KC
7    2006_01_NO_CLE     2006_01_NO_CLE
8   2006_01_DAL_JAX    2006_01_DAL_JAX
9   2006_01_DEN_STL     2006_01_DEN_LA
10  2006_01_SEA_DET    2006_01_SEA_DET
12  2006_01_PHI_HOU    2006_01_PHI_HOU
13  2006_01_IND_NYG    2006_01_IND_NYG
17   2006_01_SD_OAK     2006_01_LAC_LV
19  2006_01_MIA_PIT    2006_01_MIA_PIT
20  2006_01_MIN_WAS    2006_01_MIN_WAS
24  2006_01_NYJ_TEN    2006_01_NYJ_TEN
32  2006_02_ARI_SEA    2006_02_ARI_SEA
33   2006_02_TB_ATL     2006_02_TB_ATL
34  2006_02_OAK_BAL     2006_02_LV_BAL
35  2006_02_BUF_MIA    2006_02_BUF_MIA


In [60]:
print(
    "Matching games:",
    betting["game_id"].isin(
        team_stats["game_id_normalized"]
    ).sum()
)

print(
    "Unmatched betting games:",
    (~betting["game_id"].isin(
        team_stats["game_id_normalized"]
    )).sum()
)

Matching games: 4927
Unmatched betting games: 0


In [61]:
team_stats = team_stats.merge(
    betting,
    left_on="game_id_normalized",
    right_on="game_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", team_stats.shape)

Shape: (9854, 68)


In [62]:
print("Rows:", len(team_stats))

print(
    "Unique games:",
    team_stats["game_id_normalized"].nunique()
)

print(
    "Rows per game:"
)
print(
    team_stats
    .groupby("game_id_normalized")
    .size()
    .value_counts()
    .sort_index()
)

print(
    "\nMissing betting data:",
    team_stats["team_favorite_id"].isna().sum()
)

Rows: 9854
Unique games: 4927
Rows per game:
2    4927
Name: count, dtype: int64

Missing betting data: 0


In [63]:
print(
    team_stats["team_favorite_id"]
    .value_counts()
    .sort_index()
)

team_favorite_id
ARI     248
ATL     318
BAL     416
BUF     292
CAR     258
CHI     246
CIN     312
CLE     204
DAL     390
DEN     332
DET     248
GB      414
HOU     270
IND     352
JAX     184
KC      368
LAC     390
LAR     252
LVR     170
MIA     234
MIN     326
NE      464
NO      394
NYG     250
NYJ     236
PHI     416
PICK     40
PIT     408
SEA     358
SF      332
TB      260
TEN     278
WAS     194
Name: count, dtype: int64


In [64]:
print(
    "Unique favorite IDs:",
    sorted(team_stats["team_favorite_id"].dropna().unique())
)

Unique favorite IDs: ['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LAC', 'LAR', 'LVR', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PICK', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


In [65]:
favorite_id_map = {
    "LAR": "LA",
    "LVR": "LV"
}

team_stats["favorite_team"] = (
    team_stats["team_favorite_id"]
    .replace(favorite_id_map)
)

In [71]:
print(
    "Pick'em games:",
    team_stats.loc[
        team_stats["favorite_team"] == "PICK",
        "game_id_normalized"
    ].nunique()
)

Pick'em games: 20


In [72]:
games = (
    team_stats[
        [
            "game_id_normalized",
            "season",
            "week",
            "team_favorite_id",
            "favorite_team",
            "score_home",
            "score_away",
            "team_home",
            "team_away",
            "spread_favorite"
        ]
    ]
    .drop_duplicates("game_id_normalized")
    .copy()
)

print("Rows:", len(games))
print("Unique games:", games["game_id_normalized"].nunique())

Rows: 4927
Unique games: 4927


In [73]:
print(
    games[
        [
            "season",
            "week",
            "team_home",
            "team_away",
            "favorite_team",
            "score_home",
            "score_away",
            "spread_favorite"
        ]
    ].head(10)
)

    season  week team_home team_away favorite_team  score_home  score_away  \
0     2006     1       ARI        SF           ARI          34          27   
1     2006     1       CAR       ATL           CAR           6          20   
2     2006     1        TB       BAL            TB           0          27   
3     2006     1        NE       BUF            NE          19          17   
5     2006     1        GB       CHI           CHI           0          26   
6     2006     1        KC       CIN           CIN          10          23   
7     2006     1       CLE        NO           CLE          14          19   
8     2006     1       JAX       DAL           DAL          24          17   
9     2006     1        LA       DEN           DEN          18          10   
10    2006     1       DET       SEA           SEA           6           9   

    spread_favorite  
0              -9.5  
1              -4.5  
2              -3.0  
3             -10.0  
5              -3.5  
6        

In [75]:
games["favorite_score"] = games.apply(
    lambda row: (
        row["score_home"]
        if row["favorite_team"] == row["team_home"]
        else row["score_away"]
    ),
    axis=1
)

games["underdog_score"] = games.apply(
    lambda row: (
        row["score_away"]
        if row["favorite_team"] == row["team_home"]
        else row["score_home"]
    ),
    axis=1
)

games["favorite_margin"] = (
    games["favorite_score"] - games["underdog_score"]
)

In [76]:
games["betting_upset"] = (
    (games["favorite_team"] != "PICK") &
    (games["favorite_margin"] < 0)
)

In [77]:
print(
    games[
        [
            "season",
            "week",
            "team_home",
            "team_away",
            "favorite_team",
            "favorite_score",
            "underdog_score",
            "favorite_margin",
            "betting_upset"
        ]
    ].head(15)
)

print("\nBetting-market upsets:", games["betting_upset"].sum())

print(
    "Betting-market upset rate:",
    games["betting_upset"].mean()
)

    season  week team_home team_away favorite_team  favorite_score  \
0     2006     1       ARI        SF           ARI              34   
1     2006     1       CAR       ATL           CAR               6   
2     2006     1        TB       BAL            TB               0   
3     2006     1        NE       BUF            NE              19   
5     2006     1        GB       CHI           CHI              26   
6     2006     1        KC       CIN           CIN              23   
7     2006     1       CLE        NO           CLE              14   
8     2006     1       JAX       DAL           DAL              17   
9     2006     1        LA       DEN           DEN              10   
10    2006     1       DET       SEA           SEA               9   
12    2006     1       HOU       PHI           PHI              24   
13    2006     1       NYG       IND           IND              26   
17    2006     1        LV       LAC           LAC              27   
19    2006     1    

In [80]:
games["upset_margin"] = (
    games["favorite_margin"]
    .clip(upper=0)
    .abs()
)

games.loc[
    games["favorite_team"] == "PICK",
    "upset_margin"
] = 0

In [82]:
print(
    "Upsets:",
    games["betting_upset"].sum()
)

print(
    "Games with nonzero upset margin:",
    (games["upset_margin"] > 0).sum()
)

Upsets: 1637
Games with nonzero upset margin: 1637


In [83]:
home_results = games[
    [
        "season",
        "week",
        "team_home",
        "score_home",
        "score_away"
    ]
].copy()

home_results["wins"] = (
    home_results["score_home"] > home_results["score_away"]
).astype(int)

away_results = games[
    [
        "season",
        "week",
        "team_away",
        "score_away",
        "score_home"
    ]
].copy()

away_results["wins"] = (
    away_results["score_away"] > away_results["score_home"]
).astype(int)

home_results = home_results.rename(
    columns={"team_home": "team"}
)

away_results = away_results.rename(
    columns={"team_away": "team"}
)

In [84]:
team_results = pd.concat(
    [
        home_results[["season", "week", "team", "wins"]],
        away_results[["season", "week", "team", "wins"]]
    ],
    ignore_index=True
)

print("Rows:", len(team_results))
print("Unique teams:", team_results["team"].nunique())

Rows: 9854
Unique teams: 32


In [85]:
final_wins = (
    team_results
    .groupby(["season", "team"])["wins"]
    .sum()
    .reset_index(name="final_wins")
)

print(final_wins.head(20))

    season team  final_wins
0     2006  ARI           5
1     2006  ATL           7
2     2006  BAL          13
3     2006  BUF           7
4     2006  CAR           8
5     2006  CHI          13
6     2006  CIN           8
7     2006  CLE           4
8     2006  DAL           9
9     2006  DEN           9
10    2006  DET           3
11    2006   GB           8
12    2006  HOU           6
13    2006  IND          12
14    2006  JAX           8
15    2006   KC           9
16    2006   LA           8
17    2006  LAC          14
18    2006   LV           2
19    2006  MIA           6


In [86]:
games = games.merge(
    final_wins.rename(
        columns={
            "team": "team_home",
            "final_wins": "home_final_wins"
        }
    ),
    on=["season", "team_home"],
    how="left",
    validate="many_to_one"
)

games = games.merge(
    final_wins.rename(
        columns={
            "team": "team_away",
            "final_wins": "away_final_wins"
        }
    ),
    on=["season", "team_away"],
    how="left",
    validate="many_to_one"
)

In [88]:
games["record_upset"] = (
    (
        (games["home_final_wins"] < games["away_final_wins"]) &
        (games["score_home"] > games["score_away"])
    )
    |
    (
        (games["away_final_wins"] < games["home_final_wins"]) &
        (games["score_away"] > games["score_home"])
    )
)

In [89]:
print(
    "Record-based upsets:",
    games["record_upset"].sum()
)

print(
    "Record-based upset rate:",
    games["record_upset"].mean()
)

Record-based upsets: 1138
Record-based upset rate: 0.23097219403288005


In [90]:
print(
    games.loc[
        games["record_upset"],
        [
            "season",
            "week",
            "team_home",
            "team_away",
            "home_final_wins",
            "away_final_wins",
            "score_home",
            "score_away"
        ]
    ].head(15)
)

    season  week team_home team_away  home_final_wins  away_final_wins  \
0     2006     1       ARI        SF                5                7   
1     2006     1       CAR       ATL                8                7   
5     2006     1        KC       CIN                9                8   
7     2006     1       JAX       DAL                8                9   
8     2006     1        LA       DEN                8                9   
20    2006     2       MIN       CAR                6                8   
28    2006     2        SF        LA                7                8   
31    2006     2       PHI       NYG               10                8   
39    2006     3        NE       DEN               12                9   
41    2006     3       HOU       WAS                6                5   
43    2006     3       MIA       TEN                6                8   
47    2006     4       BAL       LAC               13               14   
49    2006     4       CAR        NO  

In [91]:
print("Missing home final wins:", games["home_final_wins"].isna().sum())
print("Missing away final wins:", games["away_final_wins"].isna().sum())

print("\nRecord upset counts by season:")
print(
    games
    .groupby("season")["record_upset"]
    .agg(["sum", "mean"])
)

print("\nRecord upset counts by week:")
print(
    games
    .groupby("week")["record_upset"]
    .agg(["sum", "mean"])
)

Missing home final wins: 0
Missing away final wins: 0

Record upset counts by season:
        sum      mean
season               
2006     64  0.250000
2007     52  0.203125
2008     58  0.226562
2009     61  0.238281
2010     65  0.253906
2011     51  0.199219
2012     55  0.214844
2013     65  0.253906
2014     50  0.195312
2015     67  0.261719
2016     62  0.242188
2017     56  0.218750
2018     64  0.250000
2019     61  0.238281
2020     53  0.207031
2021     65  0.238971
2022     67  0.247232
2023     69  0.253676
2024     53  0.194853

Record upset counts by week:
      sum      mean
week               
1      70  0.231023
2      77  0.254125
3      66  0.218543
4      72  0.254417
5      71  0.261029
6      70  0.261194
7      65  0.245283
8      60  0.224719
9      51  0.197674
10     70  0.260223
11     60  0.216606
12     50  0.170068
13     67  0.226351
14     59  0.200000
15     65  0.213816
16     68  0.223684
17     76  0.250825
18     21  0.328125


In [92]:
week1_record = games.loc[
    games["week"] == 1,
    "record_upset"
]

later_record = games.loc[
    games["week"] >= 2,
    "record_upset"
]

print("Week 1:")
print("  Upsets:", week1_record.sum())
print("  Games:", len(week1_record))
print("  Rate:", week1_record.mean())

print("\nWeeks 2–18:")
print("  Upsets:", later_record.sum())
print("  Games:", len(later_record))
print("  Rate:", later_record.mean())

print("\nDifference:")
print(
    "  Percentage points:",
    (week1_record.mean() - later_record.mean()) * 100
)

Week 1:
  Upsets: 70
  Games: 303
  Rate: 0.23102310231023102

Weeks 2–18:
  Upsets: 1068
  Games: 4624
  Rate: 0.2309688581314879

Difference:
  Percentage points: 0.005424417874311249


In [93]:
print("Missing spreads:", games["spread_favorite"].isna().sum())

print("\nSpread summary:")
print(games["spread_favorite"].describe())

print("\nUnique spread values:")
print(games["spread_favorite"].value_counts().sort_index().head(20))

print("\nPICK games:")
print(
    games.loc[
        games["favorite_team"] == "PICK",
        ["season", "week", "team_home", "team_away", "spread_favorite"]
    ].head(10)
)

Missing spreads: 0

Spread summary:
count    4927.000000
mean       -5.396895
std         3.510616
min       -26.500000
25%        -7.000000
50%        -4.000000
75%        -3.000000
max         0.000000
Name: spread_favorite, dtype: float64

Unique spread values:
spread_favorite
-26.5     1
-24.5     1
-22.5     1
-21.5     1
-21.0     1
-20.5     2
-20.0     3
-19.5     1
-19.0     1
-18.5     1
-18.0     2
-17.5     2
-17.0    14
-16.5    17
-16.0    10
-15.5     7
-15.0     6
-14.5    34
-14.0    59
-13.5    52
Name: count, dtype: int64

PICK games:
      season  week team_home team_away  spread_favorite
186     2006    13        GB       NYJ              0.0
189     2006    13       MIA       JAX              0.0
407     2007    11       DEN       TEN              0.0
432     2007    13       ARI       CLE              0.0
1152    2010     9        LV        KC              0.0
1163    2010    10       JAX       HOU              0.0
1219    2010    14       BUF       CLE          

In [94]:
games["spread_deviation"] = (
    games["favorite_margin"] + games["spread_favorite"]
)

games["spread_upset_magnitude"] = (
    -games["spread_deviation"]
)

games.loc[
    games["favorite_team"] == "PICK",
    ["spread_deviation", "spread_upset_magnitude"]
] = pd.NA

In [95]:
print(
    games[
        [
            "season",
            "week",
            "team_home",
            "team_away",
            "favorite_team",
            "spread_favorite",
            "favorite_margin",
            "spread_deviation",
            "spread_upset_magnitude",
            "betting_upset"
        ]
    ]
    .loc[games["betting_upset"]]
    .sort_values("spread_upset_magnitude", ascending=False)
    .head(15)
)

      season  week team_home team_away favorite_team  spread_favorite  \
1121    2010     7       DEN        LV           DEN             -7.0   
3772    2020    13       LAC        NE           LAC             -2.0   
2872    2017     4       HOU       TEN           TEN             -2.0   
1680    2012    10       MIA       TEN           MIA             -7.0   
2849    2017     3       JAX       BAL           BAL             -4.0   
1828    2013     3       CAR       NYG           NYG             -3.0   
995     2009    16       NYG       CAR           NYG             -8.5   
4343    2022    16        LA       DEN           DEN             -3.0   
3746    2020    12       ATL        LV            LV             -3.0   
2244    2014    14        NO       CAR            NO             -8.5   
3715    2020     9        TB        NO            TB             -4.0   
3851    2021     1        NO        GB            GB             -4.0   
2826    2017     1        LA       IND           IN

In [96]:
games["spread_upset_magnitude"] = (
    -games["spread_deviation"]
)

games.loc[
    ~games["betting_upset"],
    "spread_upset_magnitude"
] = 0

games.loc[
    games["favorite_team"] == "PICK",
    "spread_upset_magnitude"
] = 0

In [97]:
print("Betting upsets:", games["betting_upset"].sum())

print(
    "\nAverage spread upset magnitude:",
    games.loc[
        games["betting_upset"],
        "spread_upset_magnitude"
    ].mean()
)

print(
    "\nMedian spread upset magnitude:",
    games.loc[
        games["betting_upset"],
        "spread_upset_magnitude"
    ].median()
)

print(
    "\nLargest spread upsets:"
)

print(
    games.loc[
        games["betting_upset"],
        [
            "season",
            "week",
            "team_home",
            "team_away",
            "favorite_team",
            "spread_favorite",
            "favorite_margin",
            "spread_upset_magnitude"
        ]
    ]
    .sort_values("spread_upset_magnitude", ascending=False)
    .head(10)
)

Betting upsets: 1637

Average spread upset magnitude: 13.641417226634086

Median spread upset magnitude: 11.5

Largest spread upsets:
      season  week team_home team_away favorite_team  spread_favorite  \
1121    2010     7       DEN        LV           DEN             -7.0   
3772    2020    13       LAC        NE           LAC             -2.0   
2872    2017     4       HOU       TEN           TEN             -2.0   
1680    2012    10       MIA       TEN           MIA             -7.0   
2849    2017     3       JAX       BAL           BAL             -4.0   
1828    2013     3       CAR       NYG           NYG             -3.0   
995     2009    16       NYG       CAR           NYG             -8.5   
4343    2022    16        LA       DEN           DEN             -3.0   
3746    2020    12       ATL        LV            LV             -3.0   
2244    2014    14        NO       CAR            NO             -8.5   

      favorite_margin  spread_upset_magnitude  
1121          

In [98]:
comparison = pd.DataFrame({
    "Week 1": [
        games.loc[games["week"] == 1, "betting_upset"].mean(),
        games.loc[games["week"] == 1, "record_upset"].mean(),
        games.loc[
            (games["week"] == 1) & (games["betting_upset"]),
            "spread_upset_magnitude"
        ].mean()
    ],
    "Weeks 2–18": [
        games.loc[games["week"] >= 2, "betting_upset"].mean(),
        games.loc[games["week"] >= 2, "record_upset"].mean(),
        games.loc[
            (games["week"] >= 2) & (games["betting_upset"]),
            "spread_upset_magnitude"
        ].mean()
    ]
}, index=[
    "Betting upset rate",
    "Record-based upset rate",
    "Average spread upset magnitude"
])

comparison["Difference"] = (
    comparison["Week 1"] - comparison["Weeks 2–18"]
)

print(comparison)

                                   Week 1  Weeks 2–18  Difference
Betting upset rate               0.353135    0.330882    0.022253
Record-based upset rate          0.231023    0.230969    0.000054
Average spread upset magnitude  14.158879   13.605229    0.553650


In [102]:
week1 = games["week"] == 1
later = games["week"] >= 2

count = [
    games.loc[week1, "betting_upset"].sum(),
    games.loc[later, "betting_upset"].sum()
]

nobs = [
    week1.sum(),
    later.sum()
]

z_stat, p_value = proportions_ztest(count, nobs)

print("Week 1 betting upsets:", count[0], "/", nobs[0])
print("Weeks 2–18 betting upsets:", count[1], "/", nobs[1])
print("Z-statistic:", z_stat)
print("p-value:", p_value)

Week 1 betting upsets: 107 / 303
Weeks 2–18 betting upsets: 1530 / 4624
Z-statistic: 0.7966856931845391
p-value: 0.42563359499463416


In [103]:
count = [
    games.loc[week1, "record_upset"].sum(),
    games.loc[later, "record_upset"].sum()
]

nobs = [
    week1.sum(),
    later.sum()
]

z_stat, p_value = proportions_ztest(count, nobs)

print("Week 1 record-based upsets:", count[0], "/", nobs[0])
print("Weeks 2–18 record-based upsets:", count[1], "/", nobs[1])
print("Z-statistic:", z_stat)
print("p-value:", p_value)

Week 1 record-based upsets: 70 / 303
Weeks 2–18 record-based upsets: 1068 / 4624
Z-statistic: 0.002170407156676261
p-value: 0.998268266998635


In [104]:
week1_magnitude = games.loc[
    (games["week"] == 1) & (games["betting_upset"]),
    "spread_upset_magnitude"
]

later_magnitude = games.loc[
    (games["week"] >= 2) & (games["betting_upset"]),
    "spread_upset_magnitude"
]

print("Week 1:")
print(week1_magnitude.describe())

print("\nWeeks 2–18:")
print(later_magnitude.describe())

Week 1:
count    107.000000
mean      14.158879
std        8.774856
min        2.000000
25%        8.000000
50%       11.500000
75%       18.500000
max       39.000000
Name: spread_upset_magnitude, dtype: float64

Weeks 2–18:
count    1530.000000
mean       13.605229
std         7.777021
min         2.000000
25%         8.000000
50%        11.500000
75%        18.000000
max        52.000000
Name: spread_upset_magnitude, dtype: float64


In [106]:
t_stat, t_p = ttest_ind(
    week1_magnitude,
    later_magnitude,
    equal_var=False
)

u_stat, u_p = mannwhitneyu(
    week1_magnitude,
    later_magnitude,
    alternative="two-sided"
)

print("Welch's t-test:")
print("  t-statistic:", t_stat)
print("  p-value:", t_p)

print("\nMann–Whitney U test:")
print("  U-statistic:", u_stat)
print("  p-value:", u_p)

Welch's t-test:
  t-statistic: 0.6354397737482885
  p-value: 0.5263726208859018

Mann–Whitney U test:
  U-statistic: 83236.0
  p-value: 0.7701712511429457


In [114]:
count_a = [
    games.loc[week1, "betting_upset"].sum(),
    games.loc[later, "betting_upset"].sum()
]

nobs_a = [
    week1.sum(),
    later.sum()
]

ci_low, ci_high = confint_proportions_2indep(
    count1=count_a[0],
    nobs1=nobs_a[0],
    count2=count_a[1],
    nobs2=nobs_a[1],
    method="wald"
)

print("95% CI for difference in betting upset rates:")
print("Lower:", ci_low)
print("Upper:", ci_high)

95% CI for difference in betting upset rates:
Lower: -0.033244739638917234
Upper: 0.07775066081927058


In [113]:
count_b = [
    games.loc[week1, "record_upset"].sum(),
    games.loc[later, "record_upset"].sum()
]

nobs_b = [
    week1.sum(),
    later.sum()
]

ci_low_b, ci_high_b = confint_proportions_2indep(
    count1=count_b[0],
    nobs1=nobs_b[0],
    count2=count_b[1],
    nobs2=nobs_b[1],
    method="wald"
)

print("95% CI for difference in record-based upset rates:")
print("Lower:", ci_low_b)
print("Upper:", ci_high_b)

95% CI for difference in record-based upset rates:
Lower: -0.04893393565529105
Upper: 0.04904242401277727
